# **Week-16 LVC Session Notebook**

## **Problem Statement**

### **Business Context**

In modern manufacturing, predictive maintenance is becoming an essential part of industrial processes to ensure equipment uptime and minimize production losses. Machine failures can lead to significant downtime and financial loss for manufacturing facilities. In this case, a manufacturing company is trying to improve the reliability and efficiency of its predictive maintenance system for a machine tool by analyzing operational and sensor data. The company collects data from various machines in its production line, monitoring attributes such as air temperature, process temperature, rotational speed, torque, tool wear, and machine failure events.

The current challenge is the accurate prediction of machine failures using the available data, allowing for proactive intervention and avoiding unplanned downtime. By predicting machine failures in advance, the company can schedule timely maintenance, reduce costly breakdowns, and optimize machine operation.

### **Objective**

You have been hired as a data scientist to develop a machine learning model that can predict machine tool failure based on a set of operational attributes. Your task is to build a predictive model, using a suitable machine learning algorithm, that can classify whether or not a machine is likely to fail during production (class label: 1 for failure, 0 for no failure). The final goal is to deploy this model to AzureML, enabling real-time failure predictions, and create an endpoint for easy integration into the company's predictive maintenance system.

**Note**: To run the notebook, please use the Standard_A1_v2 instance. For the cluster, ensure you are using the Standard_DS11_v2 instance. These configurations will be adequate for completing this week's tasks efficiently.

### **Data Description**

The dataset consists of operational data for machine tools with the following features

- **UDI (Unique Identifier):** A unique identification number for each observation (non-predictive).
- **Type:** The machine tool type (categorical, e.g., M for Medium, L for Large).
- **Air Temperature:** Air temperature during machine operation (in Kelvin).
- **Process Temperature:** The internal process temperature during machine operation (in Kelvin).
- **Rotational Speed:** The rotational speed of the machine tool (in RPM).
- **Torque:** The torque applied to the machine tool (in Nm).
- **Tool Wear:** The wear and tear on the machine tool (in minutes).
- **Failure:** The target variable indicating whether a machine failure occurred (1: failure, 0: no failure).

## **1. AzureML Environment Setup and Data Preparation**

### **1.1 Connect to Azure Machine Learning Workspace**

In [ ]:
# Handle to the workspace
from azure.ai.ml import MLClient

# Authentication package
from azure.identity import DefaultAzureCredential
credential = DefaultAzureCredential()

In [ ]:
# Get a handle to the workspace
ml_client = MLClient(
    credential=credential,
    subscription_id="------------------",
    resource_group_name="------------------",
    workspace_name="-----------------------",
)

### **1.2 Set Up Compute Cluster**

In [ ]:
from azure.ai.ml.entities import AmlCompute

# Name assigned to the compute cluster
cpu_compute_target = "cpu-cluster"

try:
    # let's see if the compute target already exists
    cpu_cluster = ml_client.compute.get(cpu_compute_target)
    print(
        f"You already have a cluster named {cpu_compute_target}, we'll reuse it as is."
    )

except Exception:
    print("Creating a new cpu compute target...")

    # Let's create the Azure ML compute object with the intended parameters
    cpu_cluster = AmlCompute(
        name=cpu_compute_target,
        # Azure ML Compute is the on-demand VM service
        type="amlcompute",
        # VM Family
        size="Standard_DS11_v2",
        # Minimum running nodes when there is no job running
        min_instances=0,
        # Nodes in cluster
        max_instances=1,
        # How many seconds will the node running after the job termination
        idle_time_before_scale_down=180,
        # Dedicated or LowPriority. The latter is cheaper but there is a chance of job termination
        tier="Dedicated",
    )

    # Now, we pass the object to MLClient's create_or_update method
    cpu_cluster = ml_client.compute.begin_create_or_update(cpu_cluster).result()

print(
    f"AMLCompute with name {cpu_cluster.name} is created, the compute size is {cpu_cluster.size}"
)

Creating a new cpu compute target...
AMLCompute with name cpu-cluster is created, the compute size is Standard_DS11_v2


### **1.3 Register Dataset as Data Asset**

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# Path to the local dataset
local_data_path = 'machine_failure_data.csv'

# Create and register the dataset as an AzureML data asset
data_asset = Data(
    path=local_data_path,
    type=AssetTypes.URI_FILE,
    description="A dataset for predicting machine failures in manufacturing",
    name="machine-failure-data"
)

In [ ]:
ml_client.data.create_or_update(data_asset)

Data({'path': 'azureml://subscriptions/6490c64b-602a-4887-b258-36064f4cb8d4/resourcegroups/default_resourse_group/workspaces/demo_workspace/datastores/workspaceblobstore/paths/LocalUpload/0443597a6f6f3027d84887ebe69433e8/machine_failure_data.csv', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_file', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'machine-failure-data', 'description': 'A dataset for predicting machine failures in manufacturing', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/6490c64b-602a-4887-b258-36064f4cb8d4/resourceGroups/default_resourse_group/providers/Microsoft.MachineLearningServices/workspaces/demo_workspace/data/machine-failure-data/versions/2', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/tscompute/code/Users/TESTP3XHV8C5OT_1734342789061', 'creation_context': <azure.ai.ml.entities._system_data.S

### **1.4 Create and Configure Job Environment**

In [ ]:
# Create a directory for the preprocessing script
import os

src_dir_env = "./env"
os.makedirs(src_dir_env, exist_ok=True)

In [ ]:
%%writefile {src_dir_env}/conda.yml
name: sklearn-env
channels:
  - conda-forge
dependencies:
  - python=3.8
  - pip=21.2.4
  - scikit-learn=0.23.2
  - scipy=1.7.1
  - pip:
    - mlflow==2.8.1
    - azureml-mlflow==1.51.0
    - azureml-inference-server-http
    - azureml-core==1.49.0
    - cloudpickle==1.6.0

Overwriting ./env/conda.yml


In [ ]:
from azure.ai.ml.entities import Environment, BuildContext

env_docker_conda = Environment(
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04",
    conda_file="env/conda.yml",
    name="machine_learning_E2E",
    description="Environment created from a Docker image plus Conda environment.",
)
ml_client.environments.create_or_update(env_docker_conda)

Environment({'arm_type': 'environment_version', 'latest_version': None, 'image': 'mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04', 'intellectual_property': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'machine_learning_E2E', 'description': 'Environment created from a Docker image plus Conda environment.', 'tags': {}, 'properties': {'azureml.labels': 'latest'}, 'print_as_yaml': False, 'id': '/subscriptions/6490c64b-602a-4887-b258-36064f4cb8d4/resourceGroups/default_resourse_group/providers/Microsoft.MachineLearningServices/workspaces/demo_workspace/environments/machine_learning_E2E/versions/2', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/tscompute/code/Users/TESTP3XHV8C5OT_1734342789061', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x7fe7a82c20e0>, 'serialize': <msrest.serialization.Serializer object at 0x7fe7a82c34c0>, 'version': '2', 'conda_file': {'

## **2. Model Development Workflow**

### **2.1 Data Preparation**

This **Data Preparation job** is designed to process an input dataset by splitting it into two parts: one for training the model and the other for testing it. The script accepts three inputs: the location of the input data (`machine_failure.csv`), the ratio for splitting the data into training and testing sets (`test_train_ratio`), and the paths to save the resulting training (`train_data`) and testing (`test_data`) data. The script first reads the input CSV data from a data asset URI, then splits it using Scikit-learn's `train_test_split` function, and saves the two parts to the specified directories. It also logs the number of records in both the training and testing datasets using MLflow.

In [ ]:
# Create a directory for the preprocessing script
import os

src_dir_job_scripts = "./data_prep"
os.makedirs(src_dir_job_scripts, exist_ok=True)

In [ ]:
%%writefile {src_dir_job_scripts}/data_prep.py

import os
import argparse
import logging
import mlflow
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", type=str, help="Path to input data")
    parser.add_argument("--test_train_ratio", type=float, default=0.2)
    parser.add_argument("--train_data", type=str, help="Path to save train data")
    parser.add_argument("--test_data", type=str, help="Path to save test data")
    args = parser.parse_args()

    # Start MLflow Run
    mlflow.start_run()

    # Log arguments
    logging.info(f"Input data path: {args.data}")
    logging.info(f"Test-train ratio: {args.test_train_ratio}")

    # Read data
    df = pd.read_csv(args.data)

    # Encoding the categorical 'Type' column
    label_encoder = LabelEncoder()
    df['Type'] = label_encoder.fit_transform(df['Type'])

    # Log the first few rows of the dataframe
    logging.info(f"Transformed Data:\n{df.head()}")

    # Split data
    train_df, test_df = train_test_split(df, test_size=args.test_train_ratio, random_state=42)

    # Save train and test data
    os.makedirs(args.train_data, exist_ok=True)
    os.makedirs(args.test_data, exist_ok=True)
    train_df.to_csv(os.path.join(args.train_data, "data.csv"), index=False)
    test_df.to_csv(os.path.join(args.test_data, "data.csv"), index=False)

    # Log completion
    mlflow.log_metric("train_size", len(train_df))
    mlflow.log_metric("test_size", len(test_df))
    mlflow.end_run()

if __name__ == "__main__":
    main()

Overwriting ./data_prep/data_prep.py


#### **Define Data Preparation job**

For this AzureML job, we define the `command` object that takes input files and output directories, then executes the script with the provided inputs and outputs. The job runs in a pre-configured AzureML environment with the necessary libraries. The result will be two separate datasets for training and testing, ready for use in subsequent steps of the machine learning pipeline.

In [ ]:
from azure.ai.ml import command, Input, Output

step_process = command(
    name="data_preparation",
    display_name="Data preparation for training",
    description="Splits the input dataset into train and test datasets",
    inputs={
        "data": Input(type="uri_file"),
        "test_train_ratio": Input(type="number"),
    },
    outputs={
        "train_data": Output(type="uri_folder", mode="rw_mount"),
        "test_data": Output(type="uri_folder", mode="rw_mount"),
    },
    code="./data_prep",
    command="""python data_prep.py \
            --data ${{inputs.data}} \
            --test_train_ratio ${{inputs.test_train_ratio}} \
            --train_data ${{outputs.train_data}} \
            --test_data ${{outputs.test_data}}""",
    environment="AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
)

### **2.2 Training the Model**

The Model Training job is designed to train a Decision Tree classifier on the dataset that was split into training and testing sets in the previous data preparation job. This job script accepts four inputs: the path to the training data (`train_data`), the path to the testing data (`test_data`), the criterion for splitting (`criterion`, with a default value of **'gini'**), and the maximum depth of the tree (`max_depth`, which is set to None by default).

The script begins by reading the training and testing data files, then processes the data to separate features (X) and target labels (y). A Decision Tree model is initialized using the given criterion and max_depth, and it is trained using the training data. The model's performance is evaluated using the `accuracy score`. The accuracy score is logged in MLflow. A confusion matrix is generated for further insights into the model's performance. Finally, the trained model is saved and stored in the specified output location as an MLflow model. The job completes by logging the final accuracy score and ending the MLflow run.


In [ ]:
# Create a directory for the preprocessing script
import os

src_dir_job_scripts = "./model_train"
os.makedirs(src_dir_job_scripts, exist_ok=True)

In [ ]:
%%writefile {src_dir_job_scripts}/model_train.py

# Required imports for training
import mlflow
import argparse

import os
import pandas as pd

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, accuracy_score

mlflow.start_run()  # Start the MLflow experiment run

os.makedirs("./outputs", exist_ok=True)  # Create the "outputs" directory if it doesn't exist

def select_first_file(path):
    """Selects the first file in a folder, assuming there's only one file.
    Args:
        path (str): Path to the directory or file to choose.
    Returns:
        str: Full path of the selected file.
    """
    files = os.listdir(path)
    return os.path.join(path, files[0])

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train_data", type=str, help="Path to train data")
    parser.add_argument("--test_data", type=str, help="Path to test data")
    parser.add_argument('--criterion', type=str, default='gini',
                        help='The function to measure the quality of a split')
    parser.add_argument('--max_depth', type=int, default=None,
                        help='The maximum depth of the tree. If None, then nodes are expanded until all the leaves contain less than min_samples_split samples.')
    parser.add_argument("--model_output", type=str, help="Path of output model")
    args = parser.parse_args()

    # Load datasets
    train_df = pd.read_csv(select_first_file(args.train_data))
    test_df = pd.read_csv(select_first_file(args.test_data))

    # Dropping the label column and assigning it to y_train
    y_train = train_df["Failure"].values  # 'failure' is the target variable in this case study

    # Dropping the 'failure' column from train_df to get the features and converting to array for model training
    X_train = train_df.drop("Failure", axis=1).values

    # Dropping the label column and assigning it to y_test
    y_test = test_df["Failure"].values  # 'failure' is the target variable for testing

    # Dropping the 'failure' column from test_df to get the features and converting to array for model testing
    X_test = test_df.drop("Failure", axis=1).values

    # Initialize and train a decision tree classifier
    tree_model = DecisionTreeClassifier(criterion=args.criterion, max_depth=args.max_depth)
    tree_model = tree_model.fit(X_train, y_train)
    tree_predictions = tree_model.predict(X_test)

    # Accuracy is chosen as the evaluation metric for this case study as it is important to know the proportion of correctly predicted failures.

    # Compute and log accuracy score
    accuracy = accuracy_score(y_test, tree_predictions)
    print(f'Accuracy of Decision Tree classifier on test set: {accuracy:.2f}')
    # Logging the accuracy score as a metric
    mlflow.log_metric("Accuracy", float(accuracy))

    # Creating a confusion matrix for further insights into the model's performance
    cm = confusion_matrix(y_test, tree_predictions)
    print(cm)

    # Output the trained model
    mlflow.sklearn.save_model(tree_model, args.model_output)

    mlflow.end_run()  # Ending the MLflow experiment run

if __name__ == "__main__":
    main()

Overwriting ./model_train/model_train.py


#### **Define Model Training Job**

For this AzureML job, we define the `command` object that takes the paths to the training and testing data, the criterion, and max depth as inputs, and outputs the trained model. The command runs in a pre-configured AzureML environment with all the necessary libraries. The job produces a trained Decision Tree classifier model, which can be used for further evaluation in the machine learning pipeline.

In [ ]:
from azure.ai.ml import command, Input, Output

train_step = command(
    name="train_machine_failure_model",  # Name of the command step for model training
    display_name="Train Decision Tree Classifier for Machine Failure Prediction",  # Display name for the step
    description="Train a Decision Tree Classifier to predict machine failures",  # Description of the task
    inputs={  # Inputs required for the command
        "train_data": Input(type="uri_folder"),  # Path to the training data (folder with CSV file)
        "test_data": Input(type="uri_folder"),  # Path to the testing data (folder with CSV file)
        "criterion": Input(type="string", default="gini"),  # Criterion for splitting (gini or entropy)
        "max_depth": Input(type="number", default=None),  # Maximum depth of the tree
    },
    outputs={  # Outputs generated by the command
        "model_output": Output(type="mlflow_model"),
    },
    code="model_train/",  # Directory where the training script is located
    command="""python model_train.py \
            --train_data ${{inputs.train_data}} \
            --test_data ${{inputs.test_data}} \
            --criterion ${{inputs.criterion}} \
            --max_depth ${{inputs.max_depth}} \
            --model_output ${{outputs.model_output}}""",  # Command to execute
    environment="AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",  # Environment configuration for the training job
    compute="cpu-cluster",  # Compute target to be used for the job
)

### **2.3 Registering the Best Trained Model**

The **Model Registration job** is designed to take the best-trained model from the hyperparameter tuning sweep job and register it in MLflow as a versioned artifact for future use in the machine failure prediction pipeline. This job script accepts one input: the path to the trained model (model). The script begins by loading the model using the `mlflow.sklearn.load_model()` function. Afterward, it registers the model in the MLflow model registry, assigning it a descriptive name (`machine_failure_prediction_model`) and specifying an artifact path (`decision_tree_failure_classifier`) where the model artifacts will be stored. Using MLflow's log_model() function, the model is logged along with its metadata, ensuring that the model is easily trackable and retrievable for future evaluation, deployment, or retraining.

In [ ]:
# Create a directory for the preprocessing script
import os

src_dir_job_scripts = "./model_register"
os.makedirs(src_dir_job_scripts, exist_ok=True)

In [ ]:
%%writefile {src_dir_job_scripts}/model_register.py

import os
import argparse
import logging
import mlflow
import pandas as pd
from pathlib import Path

mlflow.start_run()  # Starting the MLflow experiment run

def main():
    # Argument parser setup for command line arguments
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", type=str, help="Path to the trained model")  # Path to the trained model artifact
    args = parser.parse_args()

    # Load the trained model from the provided path
    model = mlflow.sklearn.load_model(Path(args.model))

    print("Registering the best trained machine failure prediction model")

    # Register the model in the MLflow Model Registry under the name "machine_failure_prediction_model"
    mlflow.sklearn.log_model(
        sk_model=model,
        registered_model_name="machine_failure_prediction_model",  # Descriptive model name for registration
        artifact_path="decision_tree_failure_classifier"  # Path to store model artifacts
    )

    # End the MLflow run
    mlflow.end_run()

if __name__ == "__main__":
    main()

Overwriting ./model_register/model_register.py


#### **Define Model Register Job**

For this AzureML job, a `command` object is defined to execute the `model_register.py` script. It accepts the best-trained model as input, runs the script in the `AzureML-sklearn-1.0-ubuntu20.04-py38-cpu` environment, and uses the same compute cluster as the previous jobs (cpu-cluster). This job plays a crucial role in the pipeline by ensuring that the best-performing model identified during hyperparameter tuning is systematically stored and made available in the MLflow registry for further evaluation, deployment, or retraining. Integrating this job into the end-to-end pipeline automates the process of registering high-quality models, completing the model development lifecycle.

In [ ]:
from azure.ai.ml import command, Input

model_register_component = command(
    name="register_model",  # Name of the command step for predictions
    display_name="Register Model",  # Display name for the step
    description="Use the best trained model from previous job to register it as a model in MLflow",  # Description
    inputs={  # Inputs required for the command
        "model": Input(type="mlflow_model"),  # Path to the best trained model
    },
    code="model_register/",  # Directory where the prediction script is located
    command="""python model_register.py \
            --model ${{inputs.model}}""",  # Command to run the prediction script
    environment="AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",  # Environment configuration for the prediction job
    compute="cpu-cluster",  # Specify the compute target to be used for the job
)

### **2.4. Assembling the End-to-End Workflow**

The end-to-end pipeline integrates all the previously defined jobs into a seamless workflow, automating the process of data preparation, model training, hyperparameter tuning, and model registration. The pipeline is designed using Azure Machine Learning's `@pipeline` decorator, specifying the compute target and providing a detailed description of the workflow.

In [ ]:
from azure.ai.ml.sweep import Choice
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import ModelType
from azure.ai.ml.dsl import pipeline

# Assemble the pipeline by chaining the jobs
@pipeline(
    compute="cpu-cluster",  # Compute target for the pipeline
    description="Pipeline for data preparation, training, prediction, and model registration",
)
def complete_pipeline(input_data_uri, test_train_ratio, criterion, max_depth):
    # Step 1: Preprocess the data
    preprocess_step = step_process(data=input_data_uri, test_train_ratio=test_train_ratio)

    # Step 2: Train the model using preprocessed data
    training_step = train_step(train_data=preprocess_step.outputs.train_data, test_data=preprocess_step.outputs.test_data, criterion=criterion, max_depth=max_depth)

    # Define the training step with hyperparameters for tuning
    job_for_sweep = training_step(
        criterion=Choice(values=["gini", "entropy"]),
        max_depth=Choice(values=[3, 5, 10, None]),
    )

    # Define the sweep job
    sweep_job = job_for_sweep.sweep(
        compute="cpu-cluster",
        sampling_algorithm="random",
        primary_metric="Accuracy",
        goal="Maximize",
    )

    # Set the limits for the sweep job:
    # - max_total_trials: The maximum number of hyperparameter combinations to be evaluated (20 in this case).
    # - max_concurrent_trials: The maximum number of trials to run simultaneously (10 in this case) to optimize resource utilization.
    # - timeout: The maximum allowed duration for the sweep job in seconds (7200 seconds, or 2 hours).
    sweep_job.set_limits(max_total_trials=20, max_concurrent_trials=10, timeout=7200)

    # Step 3: Register the best model
    # After the sweep job, get the best model
    model_register_step = model_register_component(
        model=sweep_job.outputs.model_output,  # Best model from sweep job
    )

    # Returning outputs from all steps in the pipeline
    return {
        "pipeline_job_train_data": preprocess_step.outputs.train_data,
        "pipeline_job_test_data": preprocess_step.outputs.test_data,
        "pipeline_job_best_model": sweep_job.outputs.model_output,  # Best model from sweep job
    }

1. **Data Preparation (Preprocessing Step)**:
The pipeline starts by invoking the `step_process` job, which preprocesses the raw input data (`input_data_uri`). This step splits the dataset into training and testing sets based on the provided `test_train_ratio`. The outputs from this step include the processed training and testing datasets (`train_data` and `test_data`), which are passed as inputs to the next step.

2. **Model Training**:
The second step in the pipeline is the `train_step`, which trains a Decision Tree model using the preprocessed training data. The job uses `train_data` and `test_data` from the preprocessing step and accepts hyperparameters like `criterion` and `max_depth` to configure the model. The training step is designed to work flexibly with the parameters defined in the pipeline, allowing experimentation.

3. **Hyperparameter Tuning**:
To optimize the model's performance, a Sweep Job is defined based on the training step. The sweep job explores multiple combinations of hyperparameters (`criterion` and `max_depth`) using a random sampling algorithm. It aims to maximize the model's recall metric to ensure a high true positive rate. The job limits are set to allow a maximum of 20 trials, with up to 10 trials running concurrently, and a total timeout of 7200 seconds (2 hours). This step identifies the best combination of hyperparameters for the model.

4. **Model Registration**:
Once the sweep job completes, the best-performing model is passed to the `model_register_component`. This step registers the model in the MLflow model registry, ensuring that it is versioned and available for deployment or future experimentation. The registered model includes its metadata and is stored with a descriptive name (`machine_failure_prediction_model`).

5. **Pipeline Outputs**:
The pipeline returns key outputs for further analysis, including the locations of the training and testing datasets and the best-trained model from the sweep job. These outputs ensure traceability and provide resources for subsequent tasks like evaluation and deployment.

The pipeline is instantiated by providing the required inputs, such as the data path, test-train ratio, and initial values for hyperparameters (`criterion` and `max_depth`). It is then submitted to Azure Machine Learning for execution under the experiment name `decision_tree_training_pipeline`. Real-time logs can be streamed to monitor the pipeline's progress. Once the pipeline completes, the outputs can be accessed for verification.

In [ ]:
# The code retrieves a specific version of a registered data asset using the ml_client object.
data_path = ml_client.data.get("machine-failure-data", version="1").path

In [ ]:
# Create pipeline instance
pipeline_instance = complete_pipeline(
    input_data_uri=Input(type="uri_file", path=data_path),  # Dataset path
    test_train_ratio=0.25,  # Test-train ratio
    criterion="gini",       # Initial value for criterion
    max_depth=5             # Initial value for max depth
)

In [ ]:
# Submit the pipeline to Azure ML
pipeline_job = ml_client.jobs.create_or_update(
    pipeline_instance,
    experiment_name="decision_tree_training_pipeline"
)

# Stream the output of the job for real-time logs
ml_client.jobs.stream(pipeline_job.name)

Uploading data_prep (0.0 MBs): 100%|██████████| 1602/1602 [00:00<00:00, 19289.86it/s]


Uploading model_register (0.0 MBs): 100%|██████████| 2633/2633 [00:00<00:00, 59640.02it/s]


pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.MLFlowModelJobOutput'> and will be ignored
pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored
pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored
pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.MLFlowModelJobOutput'> and will be ignored


RunId: stoic_caravan_qkvfxrszvv
Web View: https://ml.azure.com/runs/stoic_caravan_qkvfxrszvv?wsid=/subscriptions/6490c64b-602a-4887-b258-36064f4cb8d4/resourcegroups/default_resourse_group/workspaces/demo_workspace

Streaming logs/azureml/executionlogs.txt

[2025-01-16 09:56:19Z] Submitting 1 runs, first five are: 025f3f04:64bec09c-a89d-42fe-a309-475dbc6e729e


In [ ]:
# Access pipeline outputs (optional, after job completion)
print(f"Train data location: {pipeline_job.outputs['pipeline_job_train_data']}")
print(f"Test data location: {pipeline_job.outputs['pipeline_job_test_data']}")
print(f"Best model location: {pipeline_job.outputs['pipeline_job_best_model']}")